# Lab 7 — Logistic Regression in Depth
### Predicting Heart Disease with Real Clinical Data

In previous labs we covered the *mechanics* of classification. Today you will put those ideas into practice on a real-world clinical dataset: the **Heart Disease UCI dataset** from the Cleveland Clinic.

Your goal is to build a logistic regression model that predicts whether a patient has heart disease, and to reason carefully about *which metrics matter most* in a medical context.

**Topics covered:**
- Exploratory data analysis (EDA) on real clinical data
- Feature engineering and encoding
- Logistic regression training and coefficient interpretation
- Threshold tuning and its clinical implications
- ROC-AUC analysis
- Cross-validation

Work through each **Exercise** cell. The **Answer** cell directly below contains a solution — try before you peek!

---


## 0. Setup — Run This First

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, auc,
    precision_score, recall_score, f1_score
)

# ── Load the Heart Disease UCI dataset directly from the UCI repository ──
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
col_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]
df_raw = pd.read_csv(url, header=None, names=col_names, na_values='?')
print(f"Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()


---
## Section 1 — Exploring the Data

The dataset contains 13 clinical features and one target column.

| Column | Meaning |
|---|---|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment (0–2) |
| `ca` | Number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | Thalassemia type (1 = normal, 2 = fixed defect, 3 = reversable defect) |
| `target` | Original: 0 = no disease, 1–4 = disease severity |

> **Note:** The original `target` column has values 0–4. We will binarise it: **0 = no heart disease, 1 = heart disease present**.


### Exercise 1.1 — Inspect the Data

Use `.info()` and `.describe()` to understand the dataset.
Then print the number of missing values per column.


In [ ]:
# ── Your code here ──


**Answer 1.1**

In [ ]:
df_raw.info()


In [ ]:
df_raw.describe()


In [ ]:
print("Missing values per column:")
print(df_raw.isnull().sum())


### Exercise 1.2 — Clean and Binarise the Target

1. Drop rows with any missing values.
2. Convert the `target` column so that any value > 0 becomes 1 (heart disease present) and 0 stays 0.
3. Print the new class distribution using `.value_counts()`.


In [ ]:
# ── Your code here ──


**Answer 1.2**

In [ ]:
df = df_raw.dropna().copy()
df['target'] = (df['target'] > 0).astype(int)
print("Class distribution (0 = No Disease, 1 = Disease):")
print(df['target'].value_counts())
print(f"\nClass balance: {df['target'].mean()*100:.1f}% positive")


### Exercise 1.3 — Visualise the Age Distribution by Class

Plot two overlapping histograms — one for patients *without* heart disease and one *with* — using `plt.hist()` with `alpha=0.6`.
Add a legend and axis labels.

**Think about it:** Does older age seem associated with heart disease in this dataset?


In [ ]:
# ── Your code here ──


**Answer 1.3**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df['target'] == 0]['age'], bins=20, alpha=0.6, color='steelblue', label='No Disease')
ax.hist(df[df['target'] == 1]['age'], bins=20, alpha=0.6, color='tomato', label='Heart Disease')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.set_title('Age Distribution by Heart Disease Status')
ax.legend()
plt.tight_layout()
plt.show()
# Yes — the heart disease group tends to skew slightly older, though there is a lot of overlap.


---
## Section 2 — Preprocessing

Before training, we need to:
1. Treat **categorical** features correctly.
2. **Scale** numerical features so that no single feature dominates due to its unit.


### Exercise 2.1 — Identify Categorical vs Numerical Features

The columns `cp`, `restecg`, `slope`, and `thal` look like numbers but are actually **nominal** categories.
Encode them using `pd.get_dummies()` (one-hot encoding) and drop the first dummy to avoid multicollinearity.

Store the result in a DataFrame called `df_encoded`.
Print the new shape and the first two rows.


In [ ]:
# ── Your code here ──


**Answer 2.1**

In [ ]:
cat_cols = ['cp', 'restecg', 'slope', 'thal']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
# Convert bool columns to int for cleanliness
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)
print(f"Shape after encoding: {df_encoded.shape}")
df_encoded.head(2)


### Exercise 2.2 — Train/Test Split and Scaling

1. Define `X` (all columns except `target`) and `y` (`target`).
2. Split with `test_size=0.2`, `random_state=42`, and `stratify=y`.
3. Fit a `StandardScaler` **on the training set only**, then transform both train and test sets.
4. Print the shapes of all four arrays.

**Why do we fit the scaler only on training data?**


In [ ]:
# ── Your code here ──


**Answer 2.2**

In [ ]:
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"X_train: {X_train_sc.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test_sc.shape},   y_test:  {y_test.shape}")
print()
print("We fit the scaler ONLY on training data to avoid 'data leakage'.")
print("If we included test data when computing mean/std, the model would indirectly")
print("'see' test set statistics during training — making evaluation results overly optimistic.")


---
## Section 3 — Training Logistic Regression


### Exercise 3.1 — Fit the Model

Train a `LogisticRegression` model with default parameters on the scaled training data.
Then:
- Print the training accuracy.
- Print the test accuracy.

**Think:** Is a high training accuracy alone a sign of a good model?


In [ ]:
# ── Your code here ──


**Answer 3.1**

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_sc, y_train)

train_acc = model.score(X_train_sc, y_train)
test_acc  = model.score(X_test_sc, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy    : {test_acc:.4f}")
print()
print("High training accuracy alone is NOT enough — the model could be memorising")
print("the training data (overfitting). We always care more about test accuracy.")


### Exercise 3.2 — Interpreting Coefficients

Logistic regression assigns a **coefficient** to each feature. A large positive coefficient means that feature *increases* the log-odds of heart disease; a large negative coefficient means it *decreases* it.

1. Create a `pd.Series` of coefficients, indexed by feature names.
2. Sort by absolute value (descending).
3. Plot a horizontal bar chart of the top 10 most influential features.

**Think:** Which feature has the strongest influence? Does it match your intuition?


In [ ]:
# ── Your code here ──


**Answer 3.2**

In [ ]:
coef_series = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
top10 = coef_series.head(10)
colors = ['tomato' if c > 0 else 'steelblue' for c in top10]
ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds contribution)')
ax.set_title('Top 10 Logistic Regression Coefficients')
plt.tight_layout()
plt.show()

print("\nTop 5 features by absolute coefficient:")
print(coef_series.head(5).to_string())


---
## Section 4 — Evaluation

### Exercise 4.1 — Confusion Matrix

Plot the confusion matrix for predictions made at the **default 0.5 threshold**.
Label the axes `'No Disease (0)'` and `'Heart Disease (1)'`.

From the matrix, manually identify:
- How many true positives (TP)?
- How many false negatives (FN)?


In [ ]:
# ── Your code here ──


**Answer 4.1**

In [ ]:
y_pred = model.predict(X_test_sc)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['No Disease (0)', 'Heart Disease (1)'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Threshold = 0.50')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn}  — correctly predicted No Disease")
print(f"False Positives (FP): {fp}  — predicted Disease, actually No Disease")
print(f"False Negatives (FN): {fn}  — predicted No Disease, actually Disease  ← dangerous!")
print(f"True Positives  (TP): {tp}  — correctly predicted Disease")


### Exercise 4.2 — Precision, Recall, and F1

Print the full `classification_report` for the default-threshold predictions.

Then answer these questions in comments or a markdown cell:
1. What is the Recall for the heart disease class (class 1)?
2. In a clinical screening tool, which metric is more important — Precision or Recall? Why?


In [ ]:
# ── Your code here ──


**Answer 4.2**

In [ ]:
print(classification_report(y_test, y_pred,
                           target_names=['No Disease (0)', 'Heart Disease (1)']))


**Discussion:**

- **Recall** tells us: *"Of all the patients who actually have heart disease, how many did we correctly catch?"*
- A **False Negative** (missing a real case) means a sick patient is sent home untreated — potentially life-threatening.
- A **False Positive** (flagging a healthy patient) leads to follow-up tests — inconvenient but not dangerous.

Therefore, in clinical screening, **Recall is more important than Precision**. We prefer to over-diagnose rather than miss a real case.


### Exercise 4.3 — Threshold Tuning

The default threshold is 0.5. Lower the threshold to **0.35** and repeat:
1. Generate new predictions using `predict_proba`.
2. Plot the confusion matrix.
3. Print Precision, Recall, and F1 for class 1.

Compare the results to the 0.5 threshold. What changed and why?


In [ ]:
# ── Your code here ──


**Answer 4.3**

In [ ]:
y_proba = model.predict_proba(X_test_sc)[:, 1]

threshold = 0.35
y_pred_35 = (y_proba >= threshold).astype(int)

cm35 = confusion_matrix(y_test, y_pred_35)
disp35 = ConfusionMatrixDisplay(confusion_matrix=cm35,
                                 display_labels=['No Disease (0)', 'Heart Disease (1)'])
disp35.plot(cmap='Oranges')
plt.title('Confusion Matrix — Threshold = 0.35')
plt.tight_layout()
plt.show()

p35 = precision_score(y_test, y_pred_35)
r35 = recall_score(y_test, y_pred_35)
f35 = f1_score(y_test, y_pred_35)

p50 = precision_score(y_test, y_pred)
r50 = recall_score(y_test, y_pred)
f50 = f1_score(y_test, y_pred)

print(f"{'Metric':<12} {'Threshold=0.50':>16} {'Threshold=0.35':>16}")
print("-" * 46)
print(f"{'Precision':<12} {p50:>16.4f} {p35:>16.4f}")
print(f"{'Recall':<12} {r50:>16.4f} {r35:>16.4f}")
print(f"{'F1-Score':<12} {f50:>16.4f} {f35:>16.4f}")
print()
print("Lowering the threshold → the model flags more patients as positive.")
print("→ Recall goes UP (we miss fewer real cases).")
print("→ Precision goes DOWN (more false alarms).")
print("This is the fundamental Precision–Recall trade-off.")


---
## Section 5 — ROC Curve and AUC

### Exercise 5.1 — Plot the ROC Curve

Using the raw predicted *probabilities* (not the hard labels), plot the ROC curve.

1. Compute `fpr`, `tpr`, and `thresholds` using `roc_curve`.
2. Compute the AUC using `auc(fpr, tpr)`.
3. Plot the curve with the AUC value in the legend.
4. Add a dashed diagonal line representing random guessing.

**What does an AUC of 1.0 mean? What does 0.5 mean?**


In [ ]:
# ── Your code here ──


**Answer 5.1**

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'Logistic Regression (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5,
        linestyle='--', label='Random Guessing (AUC = 0.50)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate  (1 – Specificity)')
ax.set_ylabel('True Positive Rate  (Recall / Sensitivity)')
ax.set_title('ROC Curve — Heart Disease Logistic Regression')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.4f}")
print()
print("AUC = 1.0 → perfect classifier (top-left corner of the plot).")
print("AUC = 0.5 → no better than random guessing (diagonal line).")
print("Our model's AUC tells us how well it ranks patients by risk,")
print("independent of any particular threshold choice.")


### Exercise 5.2 — Finding the Optimal Threshold on the ROC Curve

One practical way to pick a threshold is to find the point on the ROC curve that is **closest to the top-left corner** (highest TPR, lowest FPR).

1. Compute the Euclidean distance from each (fpr, tpr) point to the ideal corner (0, 1).
2. Find the threshold at that closest point.
3. Print the optimal threshold and compute Recall and Precision at that threshold.


In [ ]:
# ── Your code here ──


**Answer 5.2**

In [ ]:
# Euclidean distance from (fpr, tpr) to ideal corner (0, 1)
distances = np.sqrt(fpr**2 + (1 - tpr)**2)
best_idx  = np.argmin(distances)
best_threshold = thresholds[best_idx]

print(f"Optimal threshold (closest to top-left): {best_threshold:.4f}")
print(f"  TPR (Recall) at this point: {tpr[best_idx]:.4f}")
print(f"  FPR at this point         : {fpr[best_idx]:.4f}")

y_pred_opt = (y_proba >= best_threshold).astype(int)
print(f"\nPrecision at optimal threshold: {precision_score(y_test, y_pred_opt):.4f}")
print(f"Recall    at optimal threshold: {recall_score(y_test, y_pred_opt):.4f}")
print(f"F1-Score  at optimal threshold: {f1_score(y_test, y_pred_opt):.4f}")


---
## Section 6 — Cross-Validation

### Exercise 6.1 — 5-Fold Stratified Cross-Validation

A single train/test split can be lucky or unlucky. Cross-validation gives a more robust estimate of model performance.

1. Use `cross_val_score` with `cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
2. Score with **both** `'roc_auc'` and `'recall'`.
3. Print the individual fold scores and the mean ± standard deviation for each metric.

> **Hint:** Scale inside CV properly — fit the scaler on each fold's training portion. Here, for simplicity, use the pre-scaled `X_all_sc` defined below.


In [ ]:
# Pre-scale the entire dataset for cross-validation (demo purposes)
# In production, wrap scaler + model in a Pipeline
X_all_sc = StandardScaler().fit_transform(X)

# ── Your code here ──


**Answer 6.1**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = LogisticRegression(random_state=42, max_iter=1000)

auc_scores    = cross_val_score(cv_model, X_all_sc, y, cv=cv, scoring='roc_auc')
recall_scores = cross_val_score(cv_model, X_all_sc, y, cv=cv, scoring='recall')

print("── 5-Fold Stratified Cross-Validation ──")
print()
print("ROC-AUC per fold :", np.round(auc_scores, 4))
print(f"Mean AUC         : {auc_scores.mean():.4f}  ±  {auc_scores.std():.4f}")
print()
print("Recall per fold  :", np.round(recall_scores, 4))
print(f"Mean Recall      : {recall_scores.mean():.4f}  ±  {recall_scores.std():.4f}")
print()
print("Low standard deviation → the model generalises consistently across folds.")


---
## Section 7 — Challenge: Putting It All Together

### Challenge Exercise — Full Pipeline Report

Write a function `evaluate_model(threshold)` that:
1. Generates predictions from `y_proba` at the given threshold.
2. Prints the confusion matrix.
3. Prints Precision, Recall, F1, and Accuracy for class 1.
4. Returns a dictionary with those four values.

Then call it for thresholds `[0.30, 0.40, 0.50, 0.60]` and build a summary `pd.DataFrame` showing how each metric changes with the threshold.

Which threshold would *you* recommend for a hospital screening tool? Justify your answer.


In [ ]:
# ── Your code here ──


**Answer — Challenge**

In [ ]:
def evaluate_model(threshold):
    y_pred_t = (y_proba >= threshold).astype(int)

    cm_t = confusion_matrix(y_test, y_pred_t)
    disp_t = ConfusionMatrixDisplay(confusion_matrix=cm_t,
                                     display_labels=['No Disease', 'Heart Disease'])
    disp_t.plot(cmap='Blues')
    plt.title(f'Confusion Matrix — Threshold = {threshold}')
    plt.tight_layout()
    plt.show()

    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    a = (y_pred_t == y_test).mean()
    print(f"  Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}  Accuracy={a:.3f}\n")
    return {'threshold': threshold, 'precision': p, 'recall': r, 'f1': f, 'accuracy': a}

thresholds_to_try = [0.30, 0.40, 0.50, 0.60]
results = [evaluate_model(t) for t in thresholds_to_try]
summary = pd.DataFrame(results).set_index('threshold')
print("\n── Summary Table ──")
print(summary.round(3))


**Recommended Threshold — Discussion:**

For a **hospital screening tool**, we should prioritise **high Recall** (minimise False Negatives).
Missing a real case of heart disease is far more dangerous than an unnecessary follow-up test.

- A threshold around **0.35 – 0.40** typically achieves Recall > 0.90 while keeping Precision reasonable.
- The exact choice should involve domain experts and depends on the cost of false negatives vs. false positives in the specific clinical setting.
- The ROC curve and AUC give a threshold-independent summary of overall model quality.


---
## Summary

In this lab you:

| Step | What you did |
|---|---|
| **EDA** | Loaded real clinical data, handled missing values, binarised the target |
| **Preprocessing** | One-hot encoded categorical features, applied `StandardScaler` correctly |
| **Training** | Fit `LogisticRegression` and interpreted its coefficients |
| **Evaluation** | Read a confusion matrix, computed Precision, Recall, F1 |
| **Threshold tuning** | Saw how lowering the threshold trades Precision for Recall |
| **ROC / AUC** | Plotted the ROC curve and found the geometrically optimal threshold |
| **Cross-validation** | Estimated generalisation performance with 5-fold CV |

**Key takeaway:** In medical (and many other high-stakes) contexts, accuracy alone is misleading.
Always report Recall, Precision, and AUC — and choose your threshold deliberately based on the *cost* of each type of error.
